# Pipeline vs Barrier — Composing Sub-Agents Without Wasting Parallelism

When a task splits into independent parts, running them through one agent in sequence wastes most of
the available parallelism — and forcing every part to finish before any downstream work starts wastes
more. A small orchestration layer over sub-agents fixes both, but the two ways of composing the
stages have very different wall-clock cost, and people reach for the more expensive one far too often:

- A **barrier** composition waits for *every* sub-task in a stage to finish before the next stage
  starts. Wall-clock becomes the **sum of each stage's slowest item**.
- A **pipeline** composition lets each item flow through all its stages independently, so a fast item
  can reach the last stage while a slow one is still on the first. Wall-clock becomes the **slowest
  single chain**.

Whenever item durations are uneven — and with real documents and real model calls they always are —
the barrier's total is strictly worse, and the gap is pure idle time. A barrier is *correct* only
when the next stage genuinely needs **all** of the previous stage's results together (a merge, a
dedup, a decision over the whole set). That case is real and we build it too — but it is the
exception, not the default.

**What you'll build**

| Building block | Job |
|---|---|
| `run_barrier()` | Stage-by-stage: every item waits for the stage's slowest before moving on |
| `run_pipeline()` | Per-item chains: a fast item finishes while a slow one is still extracting |
| `Trace` + `ascii_gantt()` | Measured spans → a timeline that makes the barrier's idle time visible |
| `run_chain_safe()` | A failed sub-agent resolves to a clear empty result the batch filters out — not a crash |
| `dedupe()` + digest | The one place in this task where a barrier is genuinely right |

Everything runs on the plain Messages API with `ThreadPoolExecutor` — no framework, so the
composition logic stays inspectable.


## Setup

Sub-agent *composition*, not model choice, is the subject here, so the workers are
`claude-haiku-4-5` — fast and cheap. In production you would tier models per stage (see
[orchestrator-workers](orchestrator_workers.ipynb)); nothing about the pattern changes.


In [ ]:
%pip install --quiet --upgrade anthropic

In [1]:
import json
import threading
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field

from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY

WORKER_MODEL = "claude-haiku-4-5"

## The task: six status reports → extract → assess → brief

One realistic multi-part job, built both ways: each incoming engineering status report goes through
three sub-agent stages —

1. **extract** — pull every distinct factual claim out of the document (as a JSON array),
2. **assess** — judge risk and urgency of those facts,
3. **brief** — compress the assessment into a two-sentence executive brief.

The six documents are deliberately **uneven**: a two-line memo next to a long incident postmortem.
That unevenness is the whole story — a barrier makes five finished items wait for the sixth.


In [2]:
DOCS = {
    "doc-1": """Payments team memo: the reconciliation job finished clean for the fourth week in a
row. No action needed.""",
    "doc-2": """Search infra note: index rebuild completed 40 minutes faster after the shard
rebalancing. One warning to track: the TLS certificate for the internal gateway expires on
August 1 and renewal is not yet scheduled.""",
    "doc-3": """Mobile release update: version 9.4 rolled out to 100% of Android users; crash rate
holding at 0.11%, slightly better than 9.3. iOS review is still pending with Apple, day three of
waiting. QA flagged that the checkout API latency regression in eu-west-1 is visible in the app's
payment screen as a 1-2 second spinner. Feature flags for the loyalty program remain off pending
legal sign-off.""",
    "doc-4": """Data platform update: nightly warehouse loads are green. The new event schema
migration is 60% complete; remaining tables are owned by the growth team and blocked on their
review. We are also seeing the checkout API latency regression in eu-west-1 inflate our
end-of-day pipeline by roughly 20 minutes because retries pile up. Storage costs rose 8%
month-over-month, driven by the raw clickstream retention experiment.""",
    "doc-5": """Security review summary: quarterly access audit closed 34 of 41 findings. The seven
open items are all low severity except one: a service account with owner-level permissions on the
billing project that no current employee can explain. It has been disabled pending investigation.
Dependency scanning flagged two high-CVE packages in the notification service; patches exist and
are scheduled for this sprint. The TLS certificate for the internal gateway expires on August 1 —
security considers this the single most likely near-term outage cause. Phishing simulation
click-rate fell to 4.1% from 6.8% last quarter. The team recommends making the certificate renewal
a release blocker.""",
    "doc-6": """Incident postmortem, INC-2291, checkout API latency regression in eu-west-1.
Summary: between July 8 and July 11, p99 latency on the checkout API in eu-west-1 rose from 380ms
to 2.4s, causing an estimated 0.7% drop in completed purchases for EU traffic. Timeline: July 8,
a routine deploy enabled the new fraud-scoring sidecar for 10% of checkout traffic; July 9, the
sidecar's connection pool was found to be capped at 8 connections per pod, queueing requests under
load; July 10, a rollback was attempted but failed because the deploy pipeline's canary check
itself calls the checkout API and timed out, wedging the pipeline; July 11, manual rollback
completed and latency recovered within 20 minutes. Root cause: the sidecar shipped with a default
connection pool sized for the staging environment, and the load test suite does not exercise the
fraud-scoring path. Contributing factor: the deploy pipeline shares a failure domain with the
service it deploys — the canary check should not depend on the system being rolled back.
Follow-ups: raise the sidecar pool size and make it configuration-reviewed (owner: payments,
due July 18); add the fraud path to the load suite (owner: QA, due July 25); move canary checks
to an isolated probe service (owner: platform, due August 8); add a p99 latency alert at 800ms
for every regional checkout endpoint (owner: SRE, done). Lessons: partial rollouts hide
connection-pool exhaustion until traffic scales, and recovery tooling must not depend on the
system it recovers.""",
}

for name, doc in DOCS.items():
    print(f"{name}: {len(doc.split()):>4} words")

doc-1:   18 words
doc-2:   34 words
doc-3:   65 words
doc-4:   64 words
doc-5:  103 words
doc-6:  241 words


## Instrumentation first

You cannot argue about parallelism without a timeline. `Trace` records a `(item, stage, start, end)`
span for every sub-agent call plus token usage, and `ascii_gantt()` renders those spans as one row
per item. Gaps between letters are **idle time** — an item sitting finished, waiting for a barrier.


In [3]:
@dataclass
class Trace:
    """Thread-safe recorder of when each (item, stage) sub-agent call ran and what it cost."""

    t0: float = field(default_factory=time.monotonic)
    spans: list = field(default_factory=list)  # (item, stage, start_s, end_s) relative to t0
    tokens: dict = field(default_factory=lambda: {"in": 0, "out": 0})
    _lock: threading.Lock = field(default_factory=threading.Lock, repr=False)

    def record(self, item, stage, start, end, usage):
        with self._lock:
            self.spans.append((item, stage, start - self.t0, end - self.t0))
            self.tokens["in"] += usage.input_tokens
            self.tokens["out"] += usage.output_tokens

    def wall_clock(self):
        return max(end for *_, end in self.spans)


def agent_call(trace, item, stage, system, prompt, max_tokens=700):
    """One sub-agent = one Messages API call, timed and token-counted."""
    start = time.monotonic()
    resp = client.messages.create(
        model=WORKER_MODEL,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    trace.record(item, stage, start, time.monotonic(), resp.usage)
    return "".join(block.text for block in resp.content if block.type == "text").strip()


STAGE_MARK = {"extract": "E", "assess": "A", "brief": "B"}


def ascii_gantt(trace, width=76):
    """One row per item; letters are stages, whitespace between letters is idle time."""
    end = trace.wall_clock()
    scale = width / end
    rows = {}
    for item, stage, start_s, end_s in sorted(trace.spans):
        row = rows.setdefault(item, [" "] * width)
        lo = int(start_s * scale)
        hi = max(lo + 1, int(end_s * scale))
        for i in range(lo, min(hi, width)):
            row[i] = STAGE_MARK[stage]
    lines = [f"{item:>7} |{''.join(row)}|" for item, row in sorted(rows.items())]
    lines.append(f"{'':>8}0s{' ' * (width - 10)}{end:5.1f}s")
    return "\n".join(lines)

## The three stages

Each stage is an ordinary sub-agent call. `extract` returns a JSON array of fact strings (we will
need that structure later for the merge example); `assess` and `brief` consume whatever the previous
stage produced for **their own item only** — which is precisely why these stages do not need a
barrier.


In [4]:
EXTRACT_SYSTEM = """You extract facts from engineering status documents. Return ONLY a JSON array
of short strings, one per distinct factual claim in the document. No commentary, no code fences.
Be exhaustive: every metric, date, owner, blocker, and decision is its own fact."""

ASSESS_SYSTEM = """You are a risk analyst. Given a JSON array of facts from one status document,
return a short risk assessment: for each material risk, one line 'SEVERITY (high/medium/low): risk
— why it matters'. If nothing is risky, say 'No material risk.' Do not invent facts."""

BRIEF_SYSTEM = """You write executive briefs. Compress the given risk assessment into at most two
sentences a VP can read in ten seconds. Plain language, most urgent item first."""


def extract(trace, item, doc):
    return agent_call(trace, item, "extract", EXTRACT_SYSTEM, doc)


def assess(trace, item, facts):
    return agent_call(trace, item, "assess", ASSESS_SYSTEM, facts, max_tokens=500)


def brief(trace, item, assessment):
    return agent_call(trace, item, "brief", BRIEF_SYSTEM, assessment, max_tokens=200)


STAGES = [extract, assess, brief]

## Strategy 1 — barrier composition

Run every item's `extract` in parallel, **wait for all of them**, then every `assess`, wait again,
then every `brief`. This is the shape most people write first, because it mirrors how you'd describe
the job out loud ("first we extract everything, then we assess everything…").

The barrier is the `f.result()` loop at the end of each stage: nothing enters stage *N+1* until the
slowest item clears stage *N*.


In [5]:
def run_barrier(docs):
    trace = Trace()
    current = dict(docs)
    with ThreadPoolExecutor(max_workers=len(docs)) as pool:
        for stage_fn in STAGES:
            futures = {
                item: pool.submit(stage_fn, trace, item, value) for item, value in current.items()
            }
            current = {item: f.result() for item, f in futures.items()}  # <-- the barrier
    return current, trace


barrier_results, barrier_trace = run_barrier(DOCS)
print(ascii_gantt(barrier_trace))

  doc-1 |EEEEE            AAAAAAA           BBBB                                     |
  doc-2 |EEEEEEEE         AAAAAAA           BBBBBBBB                                 |
  doc-3 |EEEEEEEEEEE      AAAAAAAAAAAA      BBBBBBBBBBBBB                            |
  doc-4 |EEEEEEEEEE       AAAAAAAAAAAAAA    BBBBBBBBBBBBB                            |
  doc-5 |EEEEEEEEEEEEE    AAAAAAAAAAAA      BBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBBB|
  doc-6 |EEEEEEEEEEEEEEEEEAAAAAAAAAAAAAAAAAABBBBBBBBBB                               |
        0s                                                                   12.4s


Read the rows: `doc-1` (the two-line memo) finishes its `E` almost immediately — then sits idle
until `doc-6` (the postmortem) clears extraction. The same wait repeats at every stage boundary.
All that whitespace is capacity you paid for and did not use.


## Strategy 2 — pipeline composition

Same stages, same prompts, same model — the only change is *what we submit to the executor*. Instead
of submitting one stage at a time, we submit **each item's whole chain**. `doc-1` runs
extract→assess→brief back-to-back and is done while `doc-6` is still extracting. No stage boundary
exists globally; it exists only within each item.


In [6]:
def run_chain(trace, item, value):
    for stage_fn in STAGES:
        value = stage_fn(trace, item, value)
    return value


def run_pipeline(docs):
    trace = Trace()
    with ThreadPoolExecutor(max_workers=len(docs)) as pool:
        futures = {item: pool.submit(run_chain, trace, item, value) for item, value in docs.items()}
        results = {item: f.result() for item, f in futures.items()}
    return results, trace


pipeline_results, pipeline_trace = run_pipeline(DOCS)
print(ascii_gantt(pipeline_trace))

  doc-1 |EEEEEEEEEEAAAAAAABBBBBBBBBBBBBBBB                                           |
  doc-2 |EEEEEEEEEEEEEAAAAAAAAAAAAAABBBBBBBBBBBBBBB                                  |
  doc-3 |EEEEEEEEEEEEEEEEEAAAAAAAAAAAAAAAAAAAABBBBBBBBBBBBBBBBBBBBBBBBBBBB           |
  doc-4 |EEEEEEEEEEEEEEEEEEEEAAAAAAAAAAAAAAAAAAAAAAABBBBBBBBBBBBBBBB                 |
  doc-5 |EEEEEEEEEEEEEEEEEEEEEEEAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABBBBBBBBBBBBBBBB       |
  doc-6 |EEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEAAAAAAAAAAAAAAAAAAAAAAAAABBBBBBBBBBBBBBBBBBBB|
        0s                                                                    6.4s


Each row is now a solid run of letters: every item proceeds the moment *its own* previous stage
finishes. The whole batch ends when the slowest chain ends — not when three stage-wide waits have
been paid in sequence.


## The numbers

Two claims to verify against measurements, not intuition:

1. **Tokens are identical-ish** — both strategies make exactly the same 18 sub-agent calls, so cost
   in dollars is the same (any small delta is model output variance). A pipeline saves *time*, not
   tokens.
2. **Wall-clock is structurally different.** From the same per-call durations you can compute each
   strategy's floor: the barrier cannot beat the **sum of per-stage maxima**, the pipeline cannot
   beat the **slowest single chain**. The sum of maxima is always ≥ the slowest chain — equality
   only if one item is slowest at *every* stage.


In [7]:
def floors(trace):
    """Both strategies' lower bounds, computed from the same measured per-call durations."""
    dur = defaultdict(dict)
    for item, stage, start_s, end_s in trace.spans:
        dur[item][stage] = end_s - start_s
    stages_seen = {stage for _, stage, _, _ in trace.spans}
    barrier_floor = sum(max(d[stage] for d in dur.values() if stage in d) for stage in stages_seen)
    pipeline_floor = max(sum(d.values()) for d in dur.values())
    return barrier_floor, pipeline_floor


for name, trace in [("barrier", barrier_trace), ("pipeline", pipeline_trace)]:
    print(
        f"{name:>9}: wall-clock {trace.wall_clock():5.1f}s   "
        f"tokens in/out {trace.tokens['in']}/{trace.tokens['out']}"
    )

barrier_floor, pipeline_floor = floors(pipeline_trace)
print("\nfrom the pipeline run's own per-call durations:")
print(f"  barrier floor  (sum of per-stage maxima) = {barrier_floor:5.1f}s")
print(f"  pipeline floor (slowest single chain)    = {pipeline_floor:5.1f}s")
speedup = barrier_trace.wall_clock() / pipeline_trace.wall_clock()
print(f"\nobserved speedup: {speedup:.2f}x — same calls, same tokens, less waiting")

  barrier: wall-clock  12.4s   tokens in/out 3458/1897
 pipeline: wall-clock   6.4s   tokens in/out 3469/1963

from the pipeline run's own per-call durations:
  barrier floor  (sum of per-stage maxima) =   7.6s
  pipeline floor (slowest single chain)    =   6.4s

observed speedup: 1.93x — same calls, same tokens, less waiting


## The rule for choosing

Ask one question per stage boundary:

> **Does stage N need *all* of stage N−1's results together — or just each item's own result?**

| Stage N does… | Composition | Examples |
|---|---|---|
| …reads only its own item's previous output | **pipeline** (default) | extract→assess→brief above; translate→format; parse→enrich |
| …reads the whole previous stage's output | **barrier** before N | dedup/merge across items; "compare with the others"; early-exit ("0 findings → skip the expensive stage") |

And the smell test for a barrier that isn't earning its cost — if your code looks like:

```python
results = wait_for_all(stage_one)          # barrier
cleaned = [transform(r) for r in results]  # purely per-item — no cross-item dependency
final   = wait_for_all(stage_two)          # barrier again
```

…that middle transform never looks at two items at once, so the first barrier bought you nothing.
Move the transform *into* each item's chain and let the items flow.

Note the rule is **per boundary**, not per job: a real composition is usually a hybrid — per-item
flow everywhere except the one boundary that genuinely joins the set. That is the next section.


## When the barrier is right

Suppose leadership wants **one cross-document digest**, not six briefs. Three of our documents
mention the same eu-west-1 latency regression and two mention the same expiring TLS certificate — a
digest built per-item would report them five times. Deduplication *by definition* needs every
extraction result at once: this boundary earns its barrier.

Two things worth copying here:

- **The merge itself is code, not an agent.** Normalizing and comparing fact strings is a
  deterministic job; spending a sub-agent call on it adds latency, cost, and a failure mode.
- **The barrier also buys an early-exit**: with all results in hand you can decide the expensive
  final call isn't needed at all (zero facts → skip the digest).


In [8]:
def parse_facts(text):
    """Lenient JSON-array parse: strip stray code fences, fall back to line-splitting."""
    cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return [str(f).strip() for f in json.loads(cleaned)]
    except json.JSONDecodeError:
        return [line.strip("-• ").strip() for line in cleaned.splitlines() if line.strip()]


def _tokens(fact):
    return {
        w for w in "".join(c if c.isalnum() else " " for c in fact.casefold()).split() if len(w) > 2
    }


def dedupe(facts, threshold=0.6):
    """Deterministic near-duplicate filter: token-set Jaccard overlap. No agent involved."""
    unique, unique_toks, dropped = [], [], []
    for fact in facts:
        toks = _tokens(fact)
        if any(len(toks & seen) / max(1, len(toks | seen)) >= threshold for seen in unique_toks):
            dropped.append(fact)
        else:
            unique.append(fact)
            unique_toks.append(toks)
    return unique, dropped


DIGEST_SYSTEM = """You write a cross-team engineering digest. Given a deduplicated JSON array of
facts from several status documents, produce at most five bullets covering the most decision-
relevant items across ALL documents. Merge related facts; most urgent first."""

digest_trace = Trace()
with ThreadPoolExecutor(max_workers=len(DOCS)) as pool:
    futures = {item: pool.submit(extract, digest_trace, item, doc) for item, doc in DOCS.items()}
    extracted = {item: parse_facts(f.result()) for item, f in futures.items()}  # barrier: justified

all_facts = [fact for facts in extracted.values() for fact in facts]
unique, dropped = dedupe(all_facts)
print(
    f"{len(all_facts)} facts extracted -> {len(unique)} unique, {len(dropped)} near-duplicates dropped"
)
for d in dropped[:5]:
    print(f"  dropped: {d}")

if unique:  # early-exit: the barrier lets us skip the expensive call when there is nothing to say
    digest = agent_call(
        digest_trace, "digest", "brief", DIGEST_SYSTEM, json.dumps(unique), max_tokens=400
    )
    print(f"\n{digest}")
else:
    print("nothing to digest — skipped the final call entirely")

59 facts extracted -> 57 unique, 2 near-duplicates dropped
  dropped: Checkout API latency regression exists in eu-west-1
  dropped: TLS certificate for the internal gateway expires on August 1

• **URGENT: TLS certificate expiration (Aug 1) + blocked renewal** — Security flags this as the single most likely near-term outage cause. Recommend making renewal a release blocker immediately; renewal process not yet scheduled.

• **Checkout API incident aftermath requires three cross-team fixes** — Fraud-scoring sidecar shipped with undersized connection pool (staging defaults), causing 0.7% purchase drop in EU. Remediation in flight: pool size config (due 7/18), load test coverage (due 7/25), canary isolation (due 8/8); latency alerting already deployed.

• **Security finding: unexplained service account with owner-level billing access** — Account disabled pending investigation; no current employee can justify its permissions. Investigate scope and access audit impact.

• **iOS 9.4 review d

The shape to remember: **per-item flow up to the join, one barrier exactly at the join, then a
single call over the merged set.** The barrier is not a property of the job — it is a property of
one specific boundary in it.


## Result handling that does not lie

Sub-agents fail: timeouts, rate limits, an occasional refusal. In a naive composition one exception
in `f.result()` propagates up and **drops the whole batch** — five good results destroyed by one bad
item. The composition must make failure a *value* it can filter, not an event that kills it:

1. retry once (most transport failures are transient),
2. if the stage still fails, resolve that item to **`None`** — explicitly, loudly,
3. filter `None` out of the batch and report *which* items were dropped.

To keep this notebook reproducible we inject the failure deterministically — `doc-2` fails once and
is saved by the retry, `doc-4` fails permanently and is filtered. In production the same code path
is triggered by a timeout or an `overloaded_error`.


In [9]:
fault_budget = {"doc-2": 1, "doc-4": 99}  # how many times each item's assess call will blow up


def assess_flaky(trace, item, facts):
    if fault_budget.get(item, 0) > 0:
        fault_budget[item] -= 1
        raise TimeoutError(f"simulated transport timeout on assess({item})")
    return assess(trace, item, facts)


def run_chain_safe(trace, item, value, stages, retries=1):
    """A failed stage resolves the item to None instead of crashing the batch."""
    for stage_fn in stages:
        for attempt in range(retries + 1):
            try:
                value = stage_fn(trace, item, value)
                break
            except Exception as err:
                if attempt == retries:
                    print(
                        f"  {item}: {stage_fn.__name__} failed after {retries + 1} attempts "
                        f"({err}) -> resolving to None"
                    )
                    return None
    return value


safe_trace = Trace()
with ThreadPoolExecutor(max_workers=len(DOCS)) as pool:
    futures = {
        item: pool.submit(run_chain_safe, safe_trace, item, doc, [extract, assess_flaky, brief])
        for item, doc in DOCS.items()
    }
    raw = {item: f.result() for item, f in futures.items()}

survivors = {item: v for item, v in raw.items() if v is not None}
lost = sorted(set(raw) - set(survivors))
print(f"\n{len(survivors)}/{len(DOCS)} items completed; dropped: {lost}")
print(f"\nsample surviving brief ({min(survivors)}):\n{survivors[min(survivors)]}")

  doc-4: assess_flaky failed after 2 attempts (simulated transport timeout on assess(doc-4)) -> resolving to None

5/6 items completed; dropped: ['doc-4']

sample surviving brief (doc-1):
**No action required.** Reconciliation process is operating normally with consistent clean results and no identified issues.


Note what did *not* happen: no exception reached the top, `doc-2`'s transient failure was invisible
in the results (the retry absorbed it), and `doc-4`'s permanent failure cost exactly one item — with
a printed trace saying which and why. A downstream consumer sees `5/6 with doc-4 dropped`, which is
the truth, instead of a stack trace or — worse — a silently shorter list.


## The honest limit

Orchestration is not free, and this notebook should not pretend otherwise:

- **Coordination cost.** Executors, traces, retry wrappers, filters — every line above is code you
  now own, test, and debug. A single agent call has none of it.
- **Failure surface.** Six parallel chains have six independent ways to fail, plus the new failure
  modes of the composition itself (a deadlocked barrier, an unbounded queue, a filter that silently
  eats everything).
- **No parallelism, no benefit.** If your task's parts genuinely depend on each other in sequence —
  each step needs the previous step's output *of the same conversation* — a pipeline of one chain
  **is** a single agent with extra steps. Just make the calls in order.

The decision procedure, in full: *Is the work actually parallel across items? If no — one agent,
stop here. If yes — pipeline by default, and pay for a barrier only at a boundary where the next
stage reads the whole set.*


## Next steps

- [Orchestrator-Workers](orchestrator_workers.ipynb) — when the *set of sub-tasks itself* is decided
  by an LLM at runtime rather than known upfront.
- [Async Multi-Agent Orchestration](async_multi_agent_orchestration.ipynb) — when sub-agents need to
  talk to each other mid-flight, not just hand results forward.
- [Basic Workflows](basic_workflows.ipynb) — prompt chaining, routing, and parallelization as
  minimal single-file patterns.
